In [0]:
%sql
SELECT
  fonte,
  entidade,
  COUNT(*)                                              AS chamadas,
  ROUND(100 * AVG(CASE WHEN sucesso THEN 1 ELSE 0 END), 1) AS pct_sucesso,
  ROUND(AVG(latencia_ms))                               AS latencia_media_ms,
  MAX(executado_em_utc)                                 AS ultima_chamada,
  MAX_BY(erro, executado_em_utc)                        AS ultimo_erro
FROM corretora.ops.log_ingestao
WHERE executado_em_utc >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY ALL
ORDER BY pct_sucesso, fonte;

In [0]:
import os, sys
sys.path.append(os.path.abspath('..'))
from src.config import CATALOGO

# Atraso máximo aceitável, em minutos, por tabela
SLA_MINUTOS = {
    'awesomeapi_cotacoes_atuais': 60,
    'brapi_cotacoes': 60,            # só vale durante o pregão
    'clientes': 60,
    'operacoes': 60,
    'awesomeapi_historico_diario': 26 * 60,
    'awesomeapi_catalogo_pares': 26 * 60,
    'brapi_historico': 26 * 60,
    'brapi_catalogo_ativos': 26 * 60,
    'bcb_series': 26 * 60,
    'tesouro_precos_taxas': 26 * 60,
}

consultas = [
    f"SELECT '{t}' AS tabela, COUNT(*) AS registros, MAX(_data_ingestao) AS ultima_ingestao, "
    f"{sla} AS sla_minutos FROM {CATALOGO}.bronze.{t}"
    for t, sla in SLA_MINUTOS.items()
    if spark.catalog.tableExists(f'{CATALOGO}.bronze.{t}')
]
uniao = ' UNION ALL '.join(consultas)

display(spark.sql(f'''
  SELECT *,
         ROUND((unix_timestamp() - unix_timestamp(ultima_ingestao)) / 60) AS minutos_desde_ultima,
         CASE WHEN (unix_timestamp() - unix_timestamp(ultima_ingestao)) / 60 > sla_minutos
              THEN 'ATRASADA' ELSE 'OK' END AS status_frescor
  FROM ({uniao})
  ORDER BY status_frescor, tabela
'''))

In [0]:
from src.config import FONTES_BRONZE, RAW

resultado = []
for tabela, pasta, _, _ in FONTES_BRONZE:
    arquivos_raw = sum(
        1 for _, _, arquivos in os.walk(f'{RAW}/{pasta}')
        for a in arquivos if not a.startswith(('_', '.'))
    )
    nome = f'{CATALOGO}.bronze.{tabela}'
    arquivos_bronze = (spark.table(nome).select('_arquivo_origem').distinct().count()
                       if spark.catalog.tableExists(nome) else 0)
    status = 'OK' if arquivos_raw == arquivos_bronze else 'DIVERGENTE'
    resultado.append((tabela, arquivos_raw, arquivos_bronze, status))

display(spark.createDataFrame(resultado, 'tabela string, arquivos_raw long, arquivos_bronze long, status string'))

In [0]:
%sql
SELECT
  c.code                          AS ativo,
  CAST(c.bid AS DECIMAL(20, 8))   AS compra,
  CAST(c.ask AS DECIMAL(20, 8))   AS venda,
  c.create_date                   AS horario_brasilia
FROM corretora.bronze.awesomeapi_cotacoes_atuais
LATERAL VIEW explode(from_json(payload,
  'map<string, struct<code: string, bid: string, ask: string, create_date: string>>')) t AS par, c
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.code ORDER BY coletado_em_utc DESC) = 1
ORDER BY ativo;

In [0]:
%sql
WITH pontos AS (
  SELECT
    regexp_extract(url, 'sgs[.]([0-9]+)', 1) AS codigo_sgs,
    p.data,
    p.valor,
    coletado_em_utc
  FROM corretora.bronze.bcb_series
  LATERAL VIEW explode(from_json(payload, 'array<struct<data: string, valor: string>>')) t AS p
)
SELECT codigo_sgs, data, valor
FROM pontos
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY codigo_sgs
  ORDER BY to_date(data, 'dd/MM/yyyy') DESC, coletado_em_utc DESC
) = 1
ORDER BY codigo_sgs;